# Aurora Supply MaaS API test drive

**Customer question:** Can an application team discover an authorized model, make one real request with its own MaaS API key, and understand consumption and quota behavior?

Select **Kernel → Change Kernel → Aurora Inference Demo**. This notebook uses the existing `httpx` dependency; it installs nothing. Enter an **approved HTTPS MaaS endpoint**, its exact model ID, and your own scoped API key. The key is requested with a hidden prompt and kept in memory. Do not paste a key into source, print the client, inspect its attributes, or save an executed notebook with credentials.

This is a direct MaaS API exercise. It does not use the Workbench service-account token, bypass a Gateway, retrieve Aurora documents, or call inventory tools. The prompt contains synthetic historical facts; a model response is advice and must be checked by a person. No order is created.


In [ ]:
from pathlib import Path
import sys

if Path(sys.prefix).name != 'aurora-inference':
    raise RuntimeError('Select the Aurora Inference Demo kernel before running this notebook.')
helper_dir = Path.cwd() if (Path.cwd() / 'maas_workbench.py').exists() else Path.cwd() / 'notebooks'
if not (helper_dir / 'maas_workbench.py').exists():
    raise RuntimeError('Open this notebook from the rhoai-showroom checkout.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import pandas as pd
from maas_workbench import MaasClient, endpoint_config

print('Ready: verified HTTPS, no redirects, no environment proxies, and no automatic retries.')


## Choose the endpoint and model

Copy the approved **OpenAI-compatible inference URL** and **model ID** from your MaaS model details. `BASE_URL` can include the model route prefix and its trailing `/v1`; the helper adds `/v1` only when absent. Do not enter the OpenShift AI dashboard URL, the MaaS management API, URL credentials, or a query string.

The placeholder deliberately fails validation. Confirm the destination before entering a key. The client binds the key to these immutable settings; changing the variables later does not retarget it. To switch endpoints or models, edit this cell and rerun the hidden credential prompt for the new settings.


In [ ]:
BASE_URL = 'https://YOUR-MAAS-ENDPOINT.invalid'
MODEL_ID = 'YOUR-EXACT-MODEL-ID'
REQUEST_TIMEOUT_S = 20

settings = endpoint_config(BASE_URL, MODEL_ID)
print('Approved target to verify:', settings['base_url'])
print('Configured model:', settings['model_id'])


In [ ]:
from getpass import getpass

maas = MaasClient(
    settings['base_url'], settings['model_id'],
    getpass('MaaS API key for this approved endpoint (hidden): '),
    timeout_s=REQUEST_TIMEOUT_S,
)
print('Credential held in memory and bound to the selected endpoint/model.')


## Discover models, where supported

This cell sends one authenticated `GET /v1/models` request to the selected endpoint. A successful list must contain the exact configured model ID before the next cell sends inference. Some model-scoped routes return **404 or 405** for discovery; that result is shown explicitly, and you must confirm the model ID from the MaaS model details before continuing. The helper does not guess another host or follow a redirect.

A **401**, **403**, unexpected response, or transport error stops the normal test drive. Check your key's expiration, subscription, model authorization, endpoint, and TLS configuration. Do not paste credentials or raw network headers into a support request.


In [ ]:
discovery = await maas.discover_models()
discovery_summary = {key: value for key, value in discovery.items() if key != 'models'}
display(pd.DataFrame([discovery_summary]))
if discovery['models']:
    display(pd.DataFrame({'model_id': discovery['models']}))
elif discovery['outcome'] == 'discovery_unsupported':
    print('This route does not expose model discovery. Confirm MODEL_ID in the MaaS model details.')
else:
    print('No usable model list was returned. Resolve access or response issues before inference.')


## Make one actual Aurora request

The next cell sends **one** non-streaming chat completion, limited to 64 output tokens and a 20-second default total request deadline. There are no automatic retries. If discovery is unsupported, change `CONFIRM_MODEL_WITHOUT_DISCOVERY` to `True` only after checking the configured model in the MaaS model details.

Review the actual answer: the supplied historical proposal is **346 × 42 = 14,532** demo currency units and requires an **Operations manager** because the total exceeds 5,000. This is a supplied example, not a live inventory lookup. A successful HTTP request does not certify a factually correct answer.


In [ ]:
CONFIRM_MODEL_WITHOUT_DISCOVERY = False

if discovery['outcome'] == 'completed':
    if not discovery.get('configured_model_listed', False):
        raise RuntimeError('Configured MODEL_ID was not returned by this endpoint. Stop and verify it.')
elif discovery['outcome'] == 'discovery_unsupported':
    if not CONFIRM_MODEL_WITHOUT_DISCOVERY:
        raise RuntimeError('Confirm the model in MaaS details before enabling the explicit discovery exception.')
else:
    raise RuntimeError('Resolve the discovery/access failure before sending an inference request.')

reply = await maas.chat(max_tokens=64)
display(pd.DataFrame([{key: value for key, value in reply.items() if key not in ('answer', 'provider_usage')}]))
if reply['provider_usage'] is not None:
    display(pd.DataFrame([reply['provider_usage']]))
else:
    print('Provider token usage was unavailable or internally inconsistent; it is not treated as zero.')
if reply['answer'] is not None:
    print(reply['answer'])
if reply['outcome'] != 'completed':
    print('Request did not complete successfully. Stop before the optional quota exercise.')


## Relate the request to native Usage

Open **Observe & monitor → Dashboard → Usage** in OpenShift AI. Select the **Subscription** associated with the key, its **Model**, and the relevant time range; confirm the visible filters. This notebook does not assign a different subscription or override MaaS policy. Key issuance and policy determine access and quota.

- **HTTP status** is the observed client result. **Elapsed time** covers the complete non-streaming HTTP request and is not time to first token, server-only generation time, or a capacity benchmark.
- **Provider usage** is the response's consistent integer `prompt_tokens`, `completion_tokens`, and `total_tokens`; unavailable data remains unavailable.
- Native **Success rate** measures authorization/rate-limit counters, not model answer quality or all inference failures. Its query can display a 100% fallback without a usable denominator. An anonymous 401 may never reach those counters.
- **Total rate limited** needs observed counter increases. A first scrape at a nonzero value does not establish an earlier zero; one real 429 may not appear immediately. The token chart uses a rolling two-hour window, while summary panels use the selected dashboard range.

Use the existing [native dashboard guide](https://weslleyrosalem.com/rhoai-showroom/operations/native-dashboards/) and [MaaS lab](https://weslleyrosalem.com/rhoai-showroom/labs/maas/). Do not generate extra load merely to make a chart change.


## Optional: a bounded quota observation

Leave `RUN_QUOTA_PROBE = False` for the normal test drive. To demonstrate a quota, use your **authorized, deliberately small test-drive subscription** and its key. Rebind the client through the credential cell if needed. A normal subscription may complete every request without a 429; that is an honest result, not a reason to increase the load indefinitely.

The opt-in below permits at most **three sequential requests**, 32 output tokens each, two seconds between completed requests, and a 45-second overall deadline. Hard ceilings are six requests and 60 seconds. It stops at the first 429 or other error and never retries or changes policy. `Retry-After`, when present as an integer, is reported; recovery is a separate deliberate action after the applicable quota window. Do not reuse a key shared by another active demo for an exhaustion exercise.

Use **Interrupt kernel** to cancel. The helper closes the current connection and leaves no client load task running; a server may still finish a request it already accepted.


In [ ]:
RUN_QUOTA_PROBE = False
QUOTA_MAX_REQUESTS = 3
QUOTA_INTERVAL_S = 2.0
QUOTA_DURATION_S = 45

if RUN_QUOTA_PROBE:
    if reply['outcome'] != 'completed':
        raise RuntimeError('A successful single request is required before the optional quota observation.')
    quota = await maas.quota_probe(
        max_requests=QUOTA_MAX_REQUESTS,
        interval_s=QUOTA_INTERVAL_S,
        duration_s=QUOTA_DURATION_S,
    )
    print('Stop reason:', quota['stop_reason'], '| elapsed seconds:', quota['elapsed_s'])
    if quota['requests']:
        display(pd.DataFrame(quota['requests']))
else:
    print('Optional quota observation is disabled. No additional requests were sent.')


## Finish the test drive

The helper writes no reports or credentials to disk. Jupyter can persist cell outputs, so **clear all outputs before sharing or committing this notebook**. Never inspect or serialize the credential-bearing client. Restarting the kernel drops its in-memory state; this does not revoke the MaaS key. Follow your platform's key-expiration or revocation workflow when the test drive is complete.
